In [43]:
import pandas as pd
import numpy as np
import csv
import os
import glob

# Get path to repository
repo_name = "WP2_SweetProteinDiscovery"
current_path = os.getcwd()
repo_path = current_path.split(repo_name)[0] + repo_name
variants_path = f"{repo_path}/sweet_protein_variants/variants_updated.tsv"
ddg_path = f"{repo_path}/DDG_analyses"

# Read updated tsv file containing information about the variants
variants = pd.read_csv(variants_path, sep = "\t")
variants = variants.replace(' ', '-', regex=True)
variants["protein_id"] = variants["protein"] + "_" + variants["strain"]

# Replace _, _ and , characters in the mutation field
variants["mutation"] = variants["mutation"].replace('_', '; ', regex=True)
variants["mutation"] = variants["mutation"].replace('-', '', regex=True)
variants["mutation"] = variants["mutation"].replace(',', '; ', regex=True)

variants = variants.rename(columns={"Tm": "Tm_1", 
                         "Unnamed: 6": "Tm_2", 
                         "average": "Tm_average",
                         "acetonitrile 0,8%": "acetonitrile_1",
                         "Unnamed: 9": "acetonitrile_2",
                         "average.1": "acetonitrile_average",
                         "wb": "western_blot",
                        })

# Replace commas with dots in T_m and 
variants["Tm_1"] = [str(i).replace(",", ".") for i in variants["Tm_1"]]
variants["Tm_2"] = [str(i).replace(",", ".") for i in variants["Tm_2"]]
variants["Tm_average"] = [str(i).replace(",", ".") for i in variants["Tm_average"]]
variants["acetonitrile_1"] = [str(i).replace(",", ".") for i in variants["acetonitrile_1"]]
variants["acetonitrile_2"] = [str(i).replace(",", ".") for i in variants["acetonitrile_2"]]
variants["acetonitrile_average"] = [str(i).replace(",", ".") for i in variants["acetonitrile_average"]]

# Write fasta files for each mutation in a separate file in their own subfolders per protein
for index, row in variants.iterrows():
    
    protein = None
    if "Brazzein" in row["protein"]:
        protein = "Brazzein_4he7"
    elif "MNEI" in row["protein"]:
        protein = "MNEI_1iv7"
    elif "thaumatin" in row["protein"]:
        protein = "Thaumatin_3aok"
    else:
        continue
        
    try:
        os.makedirs(f'{ddg_path}/{protein}/Fastas/')
    except FileExistsError:
        pass
    
    filename = f'{ddg_path}/{protein}/Fastas/{row["protein_id"]}.fasta'  
    with open(filename, 'w', newline='\n') as file:
        writer = csv.writer(file)
        writer.writerow([">" + row["protein_id"]])
        writer.writerow([row["sequence"]])

In [44]:
# Make a column for mutation string containing mutation in the format required for DynaMut2 webserver
variants["mut_positions"] = ""
variants["dynamut_str"] = ""

for index, row in variants.iterrows():
    
    mut_str = ""
    mut_positions = []
    
    if "Brazzein" in row["protein"]:
        original = variants.iloc[0]["sequence"]
        mutation = variants.iloc[index]["sequence"]
    elif "MNEI" in row["protein"]:
        original = variants.iloc[21]["sequence"]
        mutation = variants.iloc[index]["sequence"]
    elif "thaumatin" in row["protein"]:
        original = variants.iloc[43]["sequence"]
        mutation = variants.iloc[index]["sequence"]
        
    for letter in range(0, len(original)):
        if original[letter] != mutation[letter]:
            mut_str += "A " + original[letter]+ str(letter+1) + mutation[letter] + ";"
            mut_positions.append(letter+1)
            
    variants.iloc[index, -2] = "+".join([str(i) for i in mut_positions])
    variants.iloc[index, -1] = mut_str.rstrip(";")
    
brazzein_mutations = list(variants[variants['protein'].str.contains("Brazzein")]["dynamut_str"])
thaumatin_mutations = list(variants[variants['protein'].str.contains("thaumatin")]["dynamut_str"])
monellin_mutations = list(variants[variants['protein'].str.contains("MNEI")]["dynamut_str"])

brazzein_mutations = [a for a in brazzein_mutations if len(a) > 0]
thaumatin_mutations = [a for a in thaumatin_mutations if len(a) > 0]
monellin_mutations = [a for a in monellin_mutations if len(a) > 0]

try:
    os.makedirs(f"{ddg_path}/Brazzein_4he7/Dynamut/")
except FileExistsError:
    pass

try:
    os.makedirs(f"{ddg_path}/MNEI_1iv7/Dynamut/")
except FileExistsError:
    pass

try:
    os.makedirs(f"{ddg_path}/Thaumatin_3aok/Dynamut/")
except FileExistsError:
    pass

# Save mutations as txt files
with open(f"{ddg_path}/Brazzein_4he7/Dynamut/brazzein_dynamutlist.txt", 'w') as f:
    f.write('\n'.join(brazzein_mutations))
    
with open(f"{ddg_path}/Thaumatin_3aok/Dynamut/thaumatin_dynamutlist.txt", 'w') as f:
    f.write('\n'.join(thaumatin_mutations))

with open(f"{ddg_path}/MNEI_1iv7/Dynamut/monellin_dynamutlist.txt", 'w') as f:
    f.write('\n'.join(monellin_mutations))

In [45]:
# Create _resfile.txt files that contain the mutations in the correct format for Rosetta CartesianDDG protocol
for index, row in variants.iterrows():
    
    mut_list = []
    
    if "Brazzein" in row["protein"]:
        original = variants.iloc[0]["sequence"]
        mutation = row["sequence"]
        protein = "Brazzein"
        folder = "Brazzein_4he7"
    elif "MNEI" in row["protein"]:
        original = variants.iloc[21]["sequence"]
        mutation = row["sequence"]
        protein = "Monellin"
        folder = "MNEI_1iv7"
    elif "thaumatin" in row["protein"]:
        original = variants.iloc[43]["sequence"]
        mutation = row["sequence"]
        protein= "Thaumatin"
        folder = "Thaumatin_3aok"
    
    if "native" in row["protein_id"]:
        continue

    for letter in range(0, len(original)):
        if original[letter] != mutation[letter]:
            mut_list.append(original[letter]+ " " + str(letter+1) + " " + mutation[letter])
    
    protein_id = row["protein_id"]
    with open(f'{ddg_path}/{folder}/Resfiles/{protein_id}_resfile.txt', 'w') as f:
        f.write(f'total {len(mut_list)}\n')
        f.write(f'{len(mut_list)}\n')
        f.write('\n'.join(mut_list))
        f.write('\n')

# Read CartesianDDG output files
DDG_files = glob.glob(f"{ddg_path}/*/DDG_predictions/*/*.ddg")

# Empty list for results
results = []
protein = None

# Loop over DDG output files and calculate DDG as the difference between minimum energy of mutated protein
# and relaxed PDB wild type structure: DDG = DG_MT - DG_WT. In this case, DG_MT and DG_WT are outputted by 
# Rosetta and they are given in Rosetta Energy Units (REU). Negative DDG means stabilizing mutation and vice versa.
for file in DDG_files:
    with open(file) as f:
        lines = f.readlines()

    WT_min_energy=float('inf')
    MT_min_energy=float('inf')

    for line in lines:
        sep = "fa_atr:"
        line = line.split(sep, 1)[0]
        line = line.strip()
        mut_name = line.split(":")[2].strip()
        energy = float(line.split(":")[-1])
        if "WT" in line and energy < WT_min_energy:
            WT_min_energy = energy
        elif "MUT" in line and energy < MT_min_energy:
            MT_min_energy = energy

    if 'THAUMATIN' in file.upper():
        protein = "Thaumatin"
    elif 'MNEI' in file.upper():
        protein = "Monellin"
    elif 'BRAZZEIN' in file.upper():
        protein = "Brazzein"
      
    prot_id = file.split("/")[-2].strip()
    
    ddG=MT_min_energy - WT_min_energy
    results.append([protein, prot_id, mut_name, ddG])

results_df = pd.DataFrame(results, columns = ['protein', 'protein_id', 'mutation','ddG'])
results_df = results_df.sort_values(['protein','ddG'])

In [46]:
# Extra thaumatin variants – create fasta and resfiles
extra_thaumatin_variants = pd.read_excel(f'{repo_path}/sweet_protein_variants/extra_thaumatin_variants.xlsx', header = None, names = ["Protein", "Sequence"])
protein_name = "Thaumatin_3aok" # Output folder name
native_thaumatin_sequence = variants.iloc[43]["sequence"] # Native sequence of thaumatin

# Save fastas for each variant
for index, row in extra_thaumatin_variants.iterrows():
    protein = f'{protein_name}_{row["Protein"]}'
    sequence = row["Sequence"]
    try:
        os.makedirs(f"{ddg_path}/{protein_name}/Fastas/")
    except FileExistsError:
        pass
    
    filename = f'{ddg_path}/{protein_name}/Fastas/{protein}.fasta'    
    with open(filename, 'w', newline='\n') as file:
        writer = csv.writer(file)
        writer.writerow([">" + protein])
        writer.writerow([row["Sequence"]])

# Create _resfile.txt files that contain the mutations in the correct format for Rosetta CartesianDDG protocol
mut_positions = []
for index, row in extra_thaumatin_variants.iterrows():
    mut_list = []

    # Get the mutated sequence
    mutation = row["Sequence"]

    # Compile mutation list
    for letter in range(0, len(native_thaumatin_sequence)):
        if native_thaumatin_sequence[letter] != mutation[letter]:
            mut_positions.append(int(letter+1))
            mut_list.append(native_thaumatin_sequence[letter]+ " " + str(letter+1) + " " + mutation[letter])
    
    protein = protein_name
    file_name = f'{protein_name}_{row["Protein"]}'
    os.makedirs(f'{ddg_path}/{protein}/Resfiles/', exist_ok=True)
    with open(f'{ddg_path}/{protein}/Resfiles/{file_name}_resfile.txt', 'w') as f:
        f.write(f'total {len(mut_list)}\n')
        f.write(f'{len(mut_list)}\n')
        f.write('\n'.join(mut_list))
        f.write('\n')

# Print unique mutation positions
print(set(mut_positions))

# Empty list for results
results = []
DDG_files = glob.glob(f"{ddg_path}/{protein}/DDG_predictions/*/Thaumatin_3aok*.ddg")
# Loop over DDG output files and calculate DDG as the difference between minimum energy of mutated protein
# and relaxed PDB wild type structure: DDG = DG_MT - DG_WT. In this case, DG_MT and DG_WT are outputted by 
# Rosetta and they are given in Rosetta Energy Units (REU). Negative DDG means stabilizing mutation and vice versa.

for file in DDG_files:
    with open(file) as f:
        lines = f.readlines()

    WT_min_energy=float('inf')
    MT_min_energy=float('inf')

    for line in lines:
        sep = "fa_atr:"
        line = line.split(sep, 1)[0]
        line = line.strip()
        mut_name = line.split(":")[2].strip()
        energy = float(line.split(":")[-1])
        if "WT" in line and energy < WT_min_energy:
            WT_min_energy = energy
        elif "MUT" in line and energy < MT_min_energy:
            MT_min_energy = energy
    
    prot_id = file.split("/")[-2].strip()
    ddG=MT_min_energy - WT_min_energy
    results.append([prot_id, mut_name, ddG])

# Merge results with thaumatin_i_variants
results = pd.DataFrame(results, columns=["Protein", "mutation", "ddG"])
results["SweetProtein"] = "Thaumatin_3aok"
results["Protein"] = results["Protein"].str.replace("Thaumatin_3aok_", "")
results = results.merge(extra_thaumatin_variants, on="Protein")
results.to_csv(f"{ddg_path}/extra_Thaumatin_DDG_results.csv", index=False)

{160, 65, 32, 68, 104, 105, 108, 173, 207, 26, 155}


In [47]:
# Extra thaumatin variants – create fasta and resfiles
thaumatin_variants = pd.read_excel(f'{repo_path}/sweet_protein_variants/Protein sequences for free energy calculations.xlsx')
thaumatin_variants["Protein"] = thaumatin_variants["Protein"].str.strip()
thaumatin_variants["Protein"] = thaumatin_variants["Protein"].replace(' ', '_', regex=True)
thaumatin_names = {
    "thaumatin_I": "Thaumatin_i_2vhr",
    "thaumatin_A": "Thaumatin_a_8jz8",
    "thaumatin_B": "Thaumatin_b_7lfg"
}
native_sequences = {}

# Save fastas for each variant
for index, row in thaumatin_variants.iterrows():
    
    protein = row["Protein"]
    variant = row["Protein"]
    for key in thaumatin_names.keys():
        if key == protein:
            native_sequences[protein] = row["Sequence"]
        if key in protein:
            protein = thaumatin_names[key]
    try:
        os.makedirs(f"{ddg_path}/{protein}/Fastas/")
        print(f"Created folder {protein}")
    except FileExistsError:
        pass
    
    filename = f'{ddg_path}/{protein}/Fastas/{variant}.fasta'    
    with open(filename, 'w', newline='\n') as file:
        writer = csv.writer(file)
        writer.writerow([">" + variant])
        writer.writerow([row["Sequence"]])

# Create _resfile.txt files that contain the mutations in the correct format for Rosetta CartesianDDG protocol
for protein_name in thaumatin_names.keys():
    native_sequence = native_sequences[protein_name]
    for index, row in thaumatin_variants.iterrows():
        
        mut_list = []
        if row["Protein"] == protein_name:
            continue
        elif protein_name not in row["Protein"]:
            continue

        # Get the mutated sequence
        mutation = row["Sequence"]

        # Compile mutation list
        for letter in range(0, len(native_sequence)):
            if native_sequence[letter] != mutation[letter]:
                mut_list.append(native_sequence[letter]+ " " + str(letter+1) + " " + mutation[letter])
        
        protein = thaumatin_names[protein_name]
        os.makedirs(f'{ddg_path}/{protein}/Resfiles/', exist_ok=True)
        with open(f'{ddg_path}/{protein}/Resfiles/{row["Protein"]}_resfile.txt', 'w') as f:
            f.write(f'total {len(mut_list)}\n')
            f.write(f'{len(mut_list)}\n')
            f.write('\n'.join(mut_list))
            f.write('\n')

# Empty list for results
results = []
for protein in ["Thaumatin_i_2vhr", "Thaumatin_a_8jz8", "Thaumatin_b_7lfg"]:
    DDG_files = glob.glob(f"{ddg_path}/{protein}/DDG_predictions/*/*.ddg")
    # Loop over DDG output files and calculate DDG as the difference between minimum energy of mutated protein
    # and relaxed PDB wild type structure: DDG = DG_MT - DG_WT. In this case, DG_MT and DG_WT are outputted by 
    # Rosetta and they are given in Rosetta Energy Units (REU). Negative DDG means stabilizing mutation and vice versa.
    for file in DDG_files:
        with open(file) as f:
            lines = f.readlines()

        WT_min_energy=float('inf')
        MT_min_energy=float('inf')

        for line in lines:
            sep = "fa_atr:"
            line = line.split(sep, 1)[0]
            line = line.strip()
            mut_name = line.split(":")[2].strip()
            energy = float(line.split(":")[-1])
            if "WT" in line and energy < WT_min_energy:
                WT_min_energy = energy
            elif "MUT" in line and energy < MT_min_energy:
                MT_min_energy = energy
        
        prot_id = file.split("/")[-2].strip()
    
    ddG=MT_min_energy - WT_min_energy
    results.append([prot_id, mut_name, ddG])

# Merge results with thaumatin_i_variants
results = pd.DataFrame(results, columns=["protein_id", "mutation", "ddG"])
results = results.merge(thaumatin_variants, left_on="protein_id", right_on="Protein")
results = results.drop(columns=["protein_id"])

# Remove underscores from "Protein" and replace with spaces and strip whitespace from the end
results["Protein"] = results["Protein"].replace('_', ' ', regex=True)
results["Protein"] = results["Protein"].str.strip()
results = results[["Protein", "mutation", "Sequence", "ddG"]]
results.to_csv(f"{ddg_path}/Thaumatin_DDG_results.csv", index=False)

In [49]:
brazzein_tm = float(variants[variants["protein"] == "Brazzein-native"]["Tm_average"].values[0].replace(",", "."))
monellin_tm = float(variants[variants["protein"] == "MNEI-native"]["Tm_average"].values[0].replace(",", "."))
thaumatin_tm = float(variants[variants["protein"] == "thaumatin-II-native"]["Tm_average"].values[0].replace(",", "."))
variants["delta_tm"] = None

# Calculate change in melting temperature (Delta-Tm): Tm_mutation - Tm_wildtype
# Positive delta_tm means stabilizing mutation and vice versa.  
for index, row in variants.iterrows():
    if "Brazzein" in row["protein"]:
        native_tm = brazzein_tm
    if "MNEI" in row["protein"]:
        native_tm = monellin_tm
    if "thaumatin" in row["protein"]:
        native_tm = thaumatin_tm
    
    mut_tm = row["Tm_average"]
    try: 
        delta_tm = float(mut_tm) - native_tm
    except ValueError:
        pass
    
    # Save delta_tm to the dataframe in delta_tm column
    variants.loc[index, "delta_tm"] = delta_tm

# Subset dataframe to only contain information on protein and its delta_tm
variants_tm = variants[["protein", "protein_id", "delta_tm", "dynamut_str"]].sort_values(["protein", "delta_tm"], ascending=[True,False])
variants_tm = variants_tm[["protein_id", "delta_tm", "dynamut_str"]]

# Merge results from Cartesian DDG protocol with the experimental delta_tm measurements
results_df = results_df.merge(variants_tm, how='left', on='protein_id')

thaumatin_single_dynamut = pd.read_csv(f"{ddg_path}/Thaumatin_3aok/Dynamut/Thaumatin_single_mut_results_169099668279.csv")
thaumatin_multiple_dynamut = pd.read_csv(f"{ddg_path}/Thaumatin_3aok/Dynamut/Thaumatin_multiple_mut_results_169099895464.csv")
monellin_single_dynamut = pd.read_csv(f"{ddg_path}/MNEI_1iv7/Dynamut/Monellin_single_mut_results_169099640631.csv")
monellin_multiple_dynamut = pd.read_csv(f"{ddg_path}/MNEI_1iv7/Dynamut/Monellin_multiple_mut_results_169100328832.csv")
brazzein_single_dynamut = pd.read_csv(f"{ddg_path}/Brazzein_4he7/Dynamut/Brazzein_single_mut_results_169099630467.csv")
brazzein_multiple_dynamut = pd.read_csv(f"{ddg_path}/Brazzein_4he7/Dynamut/Brazzein_multiple_mut_results_169099881928.csv")

results_df["dynamut_id"] = results_df["protein"] + "_" + results_df["dynamut_str"]
dynamut_single_values = pd.concat([thaumatin_single_dynamut,
                            monellin_single_dynamut, 
                            brazzein_single_dynamut])                           
dynamut_multiple_values = pd.concat([thaumatin_multiple_dynamut, 
                                     brazzein_multiple_dynamut,
                                     monellin_multiple_dynamut])
dynamut_multiple_values = dynamut_multiple_values.rename(columns={"mutations": "dynamut_id"})
dynamut_single_values = dynamut_single_values.rename(columns={"mutation": "dynamut_id", 
                                                              "prediction": "dynamut2_prediction"})
dynamut_values = pd.concat([dynamut_single_values, dynamut_multiple_values])
dynamut_values = dynamut_values.drop("chain", axis = 1)

merged_results_df = results_df.merge(dynamut_values, how='left', on='dynamut_id')
merged_results_df = merged_results_df.drop(["average_distance", "sum_single_predictions", "dynamut_id", "mutation"], axis = 1)
merged_results_df = merged_results_df.sort_values(["protein", "ddG"], ascending=[True,True])
merged_results_df["delta_tm"] = pd.to_numeric(merged_results_df["delta_tm"])
merged_results_df["ddG"] = pd.to_numeric(merged_results_df["ddG"])
merged_results_df.round(4).to_csv(f"{ddg_path}/DDG_results.csv", index= False)